# VSPAERO mesh quality / Cp anomaly analysis

このNotebookでは、VSPAEROが出力した`.adb`を主データとして、surface triangleの幾何品質と`Cp`を同じID上で診断する。
`.vspgeom`がある場合はalternate triangulationをoriginal NGonへ対応付け、NGon単位でも診断する。

主な確認対象は次のとおり。

- 実際の`Cp`最小値、percentile、局所`Cp` spike
- repeated global-minimum `Cp` plateauとlimiter candidate
- triangleの最小角、edge ratio、隣接面積比
- same-surface non-manifold edgeとcross-surface junction edgeの区別
- original NGonとchild triangleの対応
- wing–body junctionなど指定領域の局所診断

`Cp` spike / floor判定のthresholdは診断用heuristicであり、OpenVSPの公式validity thresholdではない。`Cp limiter candidate`もADBだけから内部limiterを断定するものではない。

## 1. importとリポジトリ位置

Notebookの実行場所から親ディレクトリを順に調べ、`src/VSPAEROMeshQuality.py`を含むリポジトリルートを見つける。


In [91]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "VSPAEROMeshQuality.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing src/VSPAEROMeshQuality.py. "
        "Run this notebook inside the repository or set repo_root manually."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.VSPAEROMeshQuality import analyze_vspaero_mesh_quality  # noqa: E402

print("repo_root:", repo_root)


repo_root: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP


## 2. 入力ファイルと診断条件

通常変更する設定はこのセルに集約する。

`.adb`と`.vspgeom`はVSPAERO実行時に生成されるためGit管理していない。既存の解析結果があるモデルを指定するか、事前にVSPAEROを実行して生成する。


In [92]:
model_name = "Boeing_777-9x_mod"
# model_name = "G103A"

model_dir = repo_root / "examples" / "models" / model_name
adb_path = model_dir / f"{model_name}.adb"
vspgeom_path = model_dir / f"{model_name}.vspgeom"
output_dir = repo_root / "examples" / "notebooks" / "mesh_quality" / "results" / model_name

solution_case = 1

# 必要な場合だけ限定する。Noneなら全surfaceを解析する。
component_ids = None
surface_ids = None
bbox = None  # (xmin, xmax, ymin, ymax, zmin, zmax)

# Local Cp spike detectionの診断用heuristic。
normal_angle_limit_deg = 30.0
spike_z_threshold = -6.0
spike_delta_threshold = -1.0

# Repeated global-minimum Cp plateauの診断用heuristic。
cp_floor_tolerance = 1e-6
cp_floor_min_cells = 4
cp_floor_max_cp = -2.0

if not adb_path.exists():
    raise FileNotFoundError(
        f"{adb_path} was not found. Run VSPAERO first so that the .adb file exists."
    )

# .vspgeomは任意。無ければtriangle-level analysisだけを実行する。
vspgeom_for_analysis = vspgeom_path if vspgeom_path.exists() else None
output_dir.mkdir(parents=True, exist_ok=True)

print("adb_path:", adb_path)
print("vspgeom_path:", vspgeom_for_analysis)
print("output_dir:", output_dir)

adb_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\models\Boeing_777-9x_mod\Boeing_777-9x_mod.adb
vspgeom_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\models\Boeing_777-9x_mod\Boeing_777-9x_mod.vspgeom
output_dir: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\notebooks\mesh_quality\results\Boeing_777-9x_mod


## 3. mesh quality / Cp解析

解析ロジックは`src/VSPAEROMeshQuality.py`に置き、このNotebookでは公開関数を呼ぶだけにする。


In [93]:
result = analyze_vspaero_mesh_quality(
    adb_path=adb_path,
    vspgeom_path=vspgeom_for_analysis,
    output_dir=output_dir,
    solution_case=solution_case,
    component_ids=component_ids,
    surface_ids=surface_ids,
    bbox=bbox,
    normal_angle_limit_deg=normal_angle_limit_deg,
    spike_z_threshold=spike_z_threshold,
    spike_delta_threshold=spike_delta_threshold,
    cp_floor_tolerance=cp_floor_tolerance,
    cp_floor_min_cells=cp_floor_min_cells,
    cp_floor_max_cp=cp_floor_max_cp,
)

## 4. summary

`cp_min_actual` / `cp_max_actual`はADB headerの表示範囲ではなく、per-triangle `Cp`から計算した実測値。
`cp_floor_plateau_detected`はrepeated global-minimum plateauの検出結果で、`n_cp_limiter_candidates`はそのplateau上の解析セル数を示す。

In [94]:
summary = pd.Series(result["summary"], name="value")
display(summary.to_frame())


,value
mesh_quality_tool_version,4
openvsp_reference_version,3.51.3
openvsp_target_version,3.51.3
adb_version,3
solution_case,1
...,...
lod_cl_abs_warning_threshold,3.0
lod_cmy_abs_warning_threshold,10.0
lod_symmetry_relative_warning_threshold,0.1
vspaero_settings,"{'Sref': 542.819, 'Cref': 9.74443, 'Bref': 71...."


## 5. local Cp spike候補

`.vspgeom`が読めた場合はoriginal NGon単位、無い場合はtriangle単位で表示する。
`cp_robust_z`だけで断定せず、`cp_local_delta`、`cp_robust_z_status`、mesh qualityを併せて確認する。
MADが0のときは有限なrobust z-scoreを作らず、`cp_robust_z_status`に理由を残す。

In [95]:
spikes = pd.DataFrame(result["spikes"])

if spikes.empty:
    print("No local Cp spikes matched the current heuristic thresholds.")
else:
    preferred_columns = [
        "ngon_id",
        "triangle_id",
        "parent_ngon_id",
        "component_id",
        "surface_id",
        "centroid_x",
        "centroid_y",
        "centroid_z",
        "cp",
        "cp_neighbor_median",
        "cp_local_delta",
        "cp_local_mad",
        "cp_robust_z",
        "cp_robust_z_status",
        "cp_zero_mad_outlier",
        "child_min_angle_deg",
        "child_max_edge_ratio",
        "child_max_neighbor_area_ratio",
        "angle_min_deg",
        "edge_ratio",
        "neighbor_area_ratio",
    ]
    columns = [column for column in preferred_columns if column in spikes.columns]
    display(spikes[columns].head(30))

No local Cp spikes matched the current heuristic thresholds.


## 6. Cp floor plateau / limiter candidate

同じ強い負圧値がglobal minimumとして複数セルに繰り返される場合を別枠で確認する。
これはVSPAERO内部のpressure limiterやvelocity capなどのsignatureになり得るが、ADBには`LocalFreeStreamVelocity(4)`が保存されないため、ここでは`limiter candidate`として扱う。

In [96]:
plateaus = pd.DataFrame(result["plateaus"])

if plateaus.empty:
    print("No repeated negative Cp floor plateau was detected.")
else:
    preferred_columns = [
        "ngon_id",
        "triangle_id",
        "parent_ngon_id",
        "component_id",
        "surface_id",
        "centroid_x",
        "centroid_y",
        "centroid_z",
        "cp",
        "cp_floor_plateau_size",
        "cp_floor_plateau_fraction",
        "cpcrit_equivalent_local_mach",
        "child_min_angle_deg",
        "child_max_edge_ratio",
        "angle_min_deg",
        "edge_ratio",
    ]
    columns = [column for column in preferred_columns if column in plateaus.columns]
    display(plateaus[columns].head(50))

,ngon_id,component_id,surface_id,centroid_x,centroid_y,centroid_z,cp,cp_floor_plateau_size,cp_floor_plateau_fraction,cpcrit_equivalent_local_mach,child_min_angle_deg,child_max_edge_ratio
0,1281,1,1,50.688291,35.574188,2.471852,-21.064816,12,0.003571,0.3,1.243865,40.427426
1,1312,1,1,50.688227,35.574188,2.472941,-21.064816,12,0.003571,0.3,1.194824,41.749528
2,1313,1,1,50.688441,35.574458,2.472195,-21.064816,12,0.003571,0.3,1.406698,40.020703
3,1314,1,1,50.660317,35.575739,2.469912,-21.064816,12,0.003571,0.3,1.375268,41.178141
4,1343,1,1,50.660205,35.575739,2.471676,-21.064816,12,0.003571,0.3,1.331671,42.472578
5,1344,1,1,50.688414,35.574458,2.472645,-21.064816,12,0.003571,0.3,1.359955,41.285418
6,2625,1,2,50.688291,-35.574188,2.471852,-21.064816,12,0.003571,0.3,1.243865,40.427426
7,2656,1,2,50.688227,-35.574188,2.472941,-21.064816,12,0.003571,0.3,1.194824,41.749528
8,2657,1,2,50.688441,-35.574458,2.472195,-21.064816,12,0.003571,0.3,1.406698,40.020703
9,2658,1,2,50.660317,-35.575739,2.469912,-21.064816,12,0.003571,0.3,1.375268,41.178141


## 7. triangle mesh qualityのworst cases

最小角、edge ratio、隣接面積比を個別に見る。単一のquality scoreにはまとめない。

In [97]:
triangles = pd.DataFrame(result["triangles"])

columns = [
    "triangle_id",
    "parent_ngon_id",
    "component_id",
    "surface_id",
    "centroid_x",
    "centroid_y",
    "centroid_z",
    "cp",
    "cp_local_delta",
    "cp_robust_z",
    "angle_min_deg",
    "edge_ratio",
    "neighbor_area_ratio",
    "max_neighbor_normal_angle_deg",
]

print("Smallest minimum angles")
display(triangles.nsmallest(30, "angle_min_deg")[columns])

print("Largest edge ratios")
display(triangles.nlargest(30, "edge_ratio")[columns])

print("Largest neighboring-area ratios")
display(triangles.nlargest(30, "neighbor_area_ratio")[columns])


Smallest minimum angles


,triangle_id,parent_ngon_id,component_id,surface_id,centroid_x,centroid_y,centroid_z,cp,cp_local_delta,cp_robust_z,angle_min_deg,edge_ratio,neighbor_area_ratio,max_neighbor_normal_angle_deg
2618,2619,1311,1,1,50.667965,35.574867,2.472611,-12.340878,4.361969,0.674491,1.179331,43.090603,4.077651,48.813101
5297,5298,2655,1,2,50.667965,-35.574867,2.472611,-10.804850,5.129983,0.674491,1.179331,43.090603,4.077651,48.813101
2619,2620,1312,1,1,50.688227,35.574188,2.472941,-21.064816,-14.771932,-1.647411,1.194824,41.749528,347.064854,47.298332
5299,5300,2656,1,2,50.688227,-35.574188,2.472941,-21.064816,-15.526402,-1.988520,1.194824,41.749528,347.064854,47.298332
2561,2562,1282,1,1,50.668130,35.574867,2.469854,-7.481407,6.791704,0.674491,1.223476,41.718394,4.049412,44.749261
5242,5243,2626,1,2,50.668130,-35.574867,2.469854,-7.943916,6.560450,0.674491,1.223476,41.718394,4.049412,44.749261
2560,2561,1281,1,1,50.688291,35.574188,2.471852,-21.064816,-17.205533,-3.203914,1.243865,40.427426,332.789163,43.117775
5240,5241,2625,1,2,50.688291,-35.574188,2.471852,-21.064816,-16.965754,-2.976249,1.243865,40.427426,332.789163,43.117775
2678,2679,1343,1,1,50.668401,35.575417,2.471507,-21.064816,0.000000,0.000000,1.331671,42.472578,3.698030,39.950794
5357,5358,2687,1,2,50.668401,-35.575417,2.471507,-21.064816,0.000000,0.000000,1.331671,42.472578,3.698030,39.950794


Largest edge ratios


,triangle_id,parent_ngon_id,component_id,surface_id,centroid_x,centroid_y,centroid_z,cp,cp_local_delta,cp_robust_z,angle_min_deg,edge_ratio,neighbor_area_ratio,max_neighbor_normal_angle_deg
2618,2619,1311,1,1,50.667965,35.574867,2.472611,-12.340878,4.361969,0.674491,1.179331,43.090603,4.077651,48.813101
5297,5298,2655,1,2,50.667965,-35.574867,2.472611,-10.804850,5.129983,0.674491,1.179331,43.090603,4.077651,48.813101
2678,2679,1343,1,1,50.668401,35.575417,2.471507,-21.064816,0.000000,0.000000,1.331671,42.472578,3.698030,39.950794
5357,5358,2687,1,2,50.668401,-35.575417,2.471507,-21.064816,0.000000,0.000000,1.331671,42.472578,3.698030,39.950794
2619,2620,1312,1,1,50.688227,35.574188,2.472941,-21.064816,-14.771932,-1.647411,1.194824,41.749528,347.064854,47.298332
5299,5300,2656,1,2,50.688227,-35.574188,2.472941,-21.064816,-15.526402,-1.988520,1.194824,41.749528,347.064854,47.298332
2561,2562,1282,1,1,50.668130,35.574867,2.469854,-7.481407,6.791704,0.674491,1.223476,41.718394,4.049412,44.749261
5242,5243,2626,1,2,50.668130,-35.574867,2.469854,-7.943916,6.560450,0.674491,1.223476,41.718394,4.049412,44.749261
2679,2680,1344,1,1,50.688414,35.574458,2.472645,-21.064816,0.000000,0.000000,1.359955,41.285418,1.118761,47.298332
5359,5360,2688,1,2,50.688414,-35.574458,2.472645,-21.064816,0.000000,0.000000,1.359955,41.285418,1.118761,47.298332


Largest neighboring-area ratios


,triangle_id,parent_ngon_id,component_id,surface_id,centroid_x,centroid_y,centroid_z,cp,cp_local_delta,cp_robust_z,angle_min_deg,edge_ratio,neighbor_area_ratio,max_neighbor_normal_angle_deg
2558,2559,1280,1,1,50.658728,35.502403,2.467062,-0.244889,10.409963,0.674491,6.505117,8.163707,347.064854,175.838631
2619,2620,1312,1,1,50.688227,35.574188,2.472941,-21.064816,-14.771932,-1.647411,1.194824,41.749528,347.064854,47.298332
5238,5239,2624,1,2,50.658728,-35.502403,2.467062,-0.271978,10.396419,0.674491,6.505117,8.163707,347.064854,175.838631
5299,5300,2656,1,2,50.688227,-35.574188,2.472941,-21.064816,-15.526402,-1.988520,1.194824,41.749528,347.064854,47.298332
2496,2497,1249,1,1,50.658765,35.502403,2.466424,-0.237159,10.413828,0.674491,6.494686,8.164274,332.789163,175.838631
2560,2561,1281,1,1,50.688291,35.574188,2.471852,-21.064816,-17.205533,-3.203914,1.243865,40.427426,332.789163,43.117775
5176,5177,2593,1,2,50.658765,-35.502403,2.466424,-0.254207,10.405304,0.674491,6.494686,8.164274,332.789163,175.838631
5240,5241,2625,1,2,50.688291,-35.574188,2.471852,-21.064816,-16.965754,-2.976249,1.243865,40.427426,332.789163,43.117775
2556,2557,1279,1,1,50.622602,35.502403,2.466891,-0.522845,0.000000,0.000000,6.585698,7.888088,97.975133,27.901616
2617,2618,1311,1,1,50.657112,35.574677,2.472897,-12.340878,-8.442746,-1.687132,4.107956,12.020212,97.975133,27.901616


## 8. NGon単位の診断

`.vspgeom`がある場合だけ実行する。`child_cp_range`は、同じparent NGonに属するchild triangle間で`Cp`が一致しているかを見るsanity checkにも使える。

In [98]:
ngons = pd.DataFrame(result["ngons"])

if ngons.empty:
    print("NGon analysis is unavailable because no .vspgeom mapping was loaded.")
else:
    columns = [
        "ngon_id",
        "component_id",
        "surface_id",
        "centroid_x",
        "centroid_y",
        "centroid_z",
        "n_vertices",
        "n_child_triangles",
        "area",
        "cp",
        "child_cp_range",
        "cp_local_delta",
        "cp_robust_z",
        "cp_robust_z_status",
        "is_cp_floor_plateau",
        "cp_floor_plateau_size",
        "child_min_angle_deg",
        "child_max_edge_ratio",
        "child_max_neighbor_area_ratio",
    ]
    print("Most negative Cp")
    display(ngons.nsmallest(30, "cp")[columns])

    print("Largest child Cp ranges")
    display(ngons.nlargest(30, "child_cp_range")[columns])

Most negative Cp


,ngon_id,component_id,surface_id,centroid_x,centroid_y,centroid_z,n_vertices,n_child_triangles,area,cp,child_cp_range,cp_local_delta,cp_robust_z,cp_robust_z_status,is_cp_floor_plateau,cp_floor_plateau_size,child_min_angle_deg,child_max_edge_ratio,child_max_neighbor_area_ratio
1280,1281,1,1,50.688291,35.574188,2.471852,3,1,0.000010,-21.064816,0.0,-17.205533,-3.203914,finite,True,12,1.243865,40.427426,332.789163
1311,1312,1,1,50.688227,35.574188,2.472941,3,1,0.000009,-21.064816,0.0,-14.771932,-1.647411,finite,True,12,1.194824,41.749528,347.064854
1312,1313,1,1,50.688441,35.574458,2.472195,3,1,0.000011,-21.064816,0.0,0.000000,0.000000,zero_mad_same,True,12,1.406698,40.020703,1.111666
1313,1314,1,1,50.660317,35.575739,2.469912,4,2,0.000052,-21.064816,0.0,-6.924269,-0.674491,finite,True,12,1.375268,41.178141,4.049412
1342,1343,1,1,50.660205,35.575739,2.471676,4,2,0.000051,-21.064816,0.0,-6.153972,-0.674491,finite,True,12,1.331671,42.472578,4.077651
1343,1344,1,1,50.688414,35.574458,2.472645,3,1,0.000010,-21.064816,0.0,0.000000,0.000000,zero_mad_same,True,12,1.359955,41.285418,1.118761
2624,2625,1,2,50.688291,-35.574188,2.471852,3,1,0.000010,-21.064816,0.0,-16.965754,-2.976249,finite,True,12,1.243865,40.427426,332.789163
2655,2656,1,2,50.688227,-35.574188,2.472941,3,1,0.000009,-21.064816,0.0,-15.526402,-1.988520,finite,True,12,1.194824,41.749528,347.064854
2656,2657,1,2,50.688441,-35.574458,2.472195,3,1,0.000011,-21.064816,0.0,0.000000,0.000000,zero_mad_same,True,12,1.406698,40.020703,1.111666
2657,2658,1,2,50.660317,-35.575739,2.469912,4,2,0.000052,-21.064816,0.0,-6.904258,-0.674491,finite,True,12,1.375268,41.178141,4.049412


Largest child Cp ranges


,ngon_id,component_id,surface_id,centroid_x,centroid_y,centroid_z,n_vertices,n_child_triangles,area,cp,child_cp_range,cp_local_delta,cp_robust_z,cp_robust_z_status,is_cp_floor_plateau,cp_floor_plateau_size,child_min_angle_deg,child_max_edge_ratio,child_max_neighbor_area_ratio
0,1,1,1,40.379578,0.560536,-3.514235,4,2,1.219432,-0.052100,0.0,0.020705,0.148815,finite,False,0,42.566693,1.474280,1.043296
1,2,1,1,39.306649,0.560559,-3.501551,4,2,1.299997,-0.166651,0.0,0.027237,0.351251,finite,False,0,45.983579,1.378597,1.043037
2,3,1,1,38.169863,0.560577,-3.557145,4,2,1.364436,-0.193888,0.0,-0.027237,-1.246909,finite,False,0,48.886456,1.305042,1.042845
3,4,1,1,36.985352,0.560587,-3.648771,4,2,1.410742,-0.151918,0.0,-0.009316,-0.252324,finite,False,0,51.017640,1.253586,1.042725
4,5,1,1,35.769230,0.560597,-3.734785,4,2,1.436567,-0.117700,0.0,-0.012840,-0.728548,finite,False,0,52.319162,1.220277,1.042623
5,6,1,1,34.540383,0.560604,-3.781363,4,2,1.440343,-0.092972,0.0,-0.010299,-0.734443,finite,False,0,52.755699,1.202809,1.056708
6,7,1,1,33.321095,0.560607,-3.769062,4,2,1.421011,-0.073215,0.0,-0.003953,-0.128132,finite,False,0,52.350098,1.199239,1.074919
7,8,1,1,32.134342,0.560606,-3.698406,4,2,1.378177,-0.048452,0.0,-0.001741,-0.044300,finite,False,0,51.165284,1.207932,1.093875
8,9,1,1,30.999924,0.560597,-3.584895,4,2,1.313537,-0.015595,0.0,-0.000705,-0.014162,finite,False,0,49.226102,1.229716,1.111474
9,10,1,1,29.933352,0.560586,-3.443859,4,2,1.232239,0.023152,0.0,0.001002,0.017898,finite,False,0,46.489639,1.270530,1.127025


## 9. mapping / topology確認

`.vspgeom`との対応失敗やsame-surface non-manifold edgeがあれば、Cpの解釈より先にgeometry / topologyを確認する。
`cross_surface_junction_edges`はHybridのThick–Thin junctionを含み得るため、それ自体をmesh failureとは扱わない。

In [99]:
print("mapping:")
display(result["mapping"])

topology = result["topology"]
print("topology:")
print("boundary_edge_count:", topology["boundary_edge_count"])
print("edge_multiplicity_gt2_count:", len(topology["edge_multiplicity_gt2"]))
print("same_surface_non_manifold_edge_count:", len(topology["same_surface_non_manifold_edges"]))
print("cross_surface_junction_edge_count:", len(topology["cross_surface_junction_edges"]))

if topology["same_surface_non_manifold_edges"]:
    print("Same-surface non-manifold candidates")
    display(pd.DataFrame(topology["same_surface_non_manifold_edges"]).head(30))

if topology["cross_surface_junction_edges"]:
    print("Cross-surface junction edges")
    display(pd.DataFrame(topology["cross_surface_junction_edges"]).head(30))

mapping:


{'mapped_triangles': 6672,
 'vspgeom_alternate_triangles': 6672,
 'mismatches': [],
 'passed': True}

topology:
boundary_edge_count: 414
edge_multiplicity_gt2_count: 0
same_surface_non_manifold_edge_count: 0
cross_surface_junction_edge_count: 0


## 10. 出力ファイル

解析結果は再確認しやすいようにCSV / JSONにも保存する。`cp_spikes.csv`と`cp_plateaus.csv`は別々に出力する。

In [100]:
for path in sorted(output_dir.glob("*")):
    print(path.relative_to(repo_root))


examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\adb_trailing_vortex_edges.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\cp_force_components.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\cp_plateaus.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\cp_spikes.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\junction_quality.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\kutta_lines.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\kutta_nodes.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\kutta_surfaces.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\kutta_symmetry.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\kutta_te_near_miss_edges.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\lod_loads.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\lod_outliers.csv
examples\notebooks\mesh_quality\results\Boeing_777-9x_mod\lod_symmetr